# CHUYÊN ĐỀ 2: BIỂU DIỄN VECTOR LIÊN TỤC VÀ CONDITIONAL FLOW MATCHING (CFM) TRONG S2ST

Notebook này giải quyết 2 trọng tâm công nghệ SOTA (State-of-the-Art):
1. **Giải mã băn khoăn "LLM học biểu diễn vector liên tục"**: Tại sao không lượng tử hóa mà giữ nguyên continuous features từ Whisper/Conformer? (SALMONN, Qwen-Audio, SLAM-Omni, LLaMA-Omni).
2. **Trường phái Whisper + Conditional Flow Matching (CFM) / Diffusion**: Voicebox, F5-TTS, CosyVoice, UNISS. Giải mã tại sao CFM (16-32 bước ODE) thay thế hoàn toàn DDPM truyền thống (1000 bước).

> 💡 **Khả năng tương thích**: Notebook này có thể chạy trực tiếp trong VS Code / Jupyter hoặc chuyển đổi sang **Marimo** (`marimo convert ... -o ...py`) để chạy trên MoLab.


## 1. Giải mã băn khoăn: "LLM học biểu diễn vector liên tục"

### Xuất xứ học thuật:
- **Paper gốc**:
  - *SALMONN (Tang et al., 2023)* - *"SALMONN: Towards Generic Hearing Abilities for Large Language Models"*.
  - *Qwen-Audio (Chu et al., Alibaba 2023)* - *"Qwen-Audio: Advancing Universal Audio Understanding via Large-Scale Audio-Language Models"*.
  - *LLaMA-Omni / SLAM-Omni (Fang et al., 2024)* - *"LLaMA-Omni: Seamless Speech Interaction with Large Language Models"*.
- **Các bài báo cùng tư duy trong thư mục của bạn**: `SLAM_Omni (1).pdf`, `S2ST_Omni Hierarchical LanguageAware SpeechLLM Adaptation for Multilingual S2ST.pdf`.

### Bản chất toán học:
1. LLM vốn dĩ hoạt động trên không gian vector liên tục $h \in \mathbb{R}^{d_{model}}$. Nó chỉ biến thành rời rạc ở tầng `lm_head` cuối cùng khi tính Softmax qua Vocabulary.
2. Whisper Encoder xuất ra chuỗi vector liên tục $H \in \mathbb{R}^{T 	imes D_{whisper}}$ ở tần số 50Hz.
3. Thay vì lượng tử hóa làm mất mát thông tin, ta dùng một **Continuous Modality Projector** (Conv1D temporal downsampler + MLP) để chiếu trực tiếp $H$ vào không gian embedding của LLM.
4. Các vector này hoạt động như các **"Soft Prompts" (Continuous Prefix)**, LLM tiếp nhận mượt mà như word embedding của văn bản mà không bị rào cản từ vựng rời rạc!


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class ContinuousWhisperProjector(nn.Module):
    """
    Lấy cảm hứng từ SALMONN (2023) và LLaMA-Omni (2024).
    Nén thời gian (temporal downsampling) từ 50Hz xuống 12.5Hz (giảm 4 lần)
    và chiếu vector liên tục của Whisper sang không gian ẩn của LLM.
    """
    def __init__(self, audio_dim=768, llm_dim=1024, downsample_rate=4):
        super().__init__()
        # Conv1D vừa giảm độ dài chuỗi vừa học đặc trưng cục bộ
        self.conv = nn.Sequential(
            nn.Conv1d(audio_dim, llm_dim, kernel_size=downsample_rate, stride=downsample_rate),
            nn.GELU(),
            nn.BatchNorm1d(llm_dim)
        )
        # MLP 2 tầng ánh xạ phi tuyến
        self.mlp = nn.Sequential(
            nn.Linear(llm_dim, llm_dim),
            nn.SiLU(),
            nn.Linear(llm_dim, llm_dim)
        )
        
    def forward(self, whisper_features):
        """
        Input:  [Batch, T_audio, audio_dim] (50Hz continuous features)
        Output: [Batch, T_audio // downsample_rate, llm_dim] (Soft Prompts cho LLM)
        """
        x = whisper_features.transpose(1, 2) # [B, audio_dim, T]
        x = self.conv(x).transpose(1, 2)     # [B, T // 4, llm_dim]
        return self.mlp(x)

# Thử nghiệm Projector
projector = ContinuousWhisperProjector(audio_dim=768, llm_dim=1024, downsample_rate=4).to(device)
fake_whisper = torch.randn(2, 100, 768, device=device) # 100 frames (~2 giây âm thanh)
continuous_tokens = projector(fake_whisper)

print("Kích thước đầu vào từ Whisper:       ", fake_whisper.shape)
print("Kích thước sau khi chiếu vào LLM:   ", continuous_tokens.shape)
print("-> Chiều dài chuỗi giảm 4 lần, tiết kiệm 75% bộ nhớ Self-Attention của LLM!")


## 2. Trường phái Whisper + Conditional Flow Matching (CFM)

### Xuất xứ học thuật:
- **Paper gốc**:
  - *Voicebox (Le et al., Meta AI 2023)* - *"Voicebox: Text-Guided Multilingual Universal Speech Generation at Scale"*.
  - *F5-TTS (Chen et al., 2024)* - *"F5-TTS: A Fairytaler that Fakes Fluent and Faithful Speech with Flow Matching"*.
  - *CosyVoice (Du et al., Alibaba 2024)* - *"CosyVoice: A Multi-lingual Large Voice Generation Model"*.
  - *UniS2ST / UNISS (ICLR 2024)* - *"Unified Expressive Speech-to-Speech Translation With Your Voice"*.
- **Các bài báo cùng tư duy trong thư mục của bạn**: `Voicebox.pdf`, `F5-TTS.pdf`, `ICLR-2026-uniss...pdf`.

### Tại sao Flow Matching (CFM) lại vượt trội hoàn toàn so với DDPM?
1. **Diffusion truyền thống (DDPM)**:
   - Mô phỏng quá trình khuếch tán Brownian ngẫu nhiên (SDE). Đường đi từ nhiễu $x_0$ đến Mel $x_1$ bị uốn lượn ngoằn ngoèo, đòi hỏi **100 đến 1000 bước** đảo ngược thời gian.
2. **Conditional Flow Matching (Optimal Transport CFM)**:
   - Thiết lập **đường thẳng ngắn nhất** nối trực tiếp giữa nhiễu Gaussian $x_0 \sim \mathcal{N}(0, I)$ và Mel-spectrogram mục tiêu $x_1$:
     $$x_t = (1 - t)x_0 + t x_1, \quad t \in [0, 1]$$
   - Đạo hàm vận tốc mục tiêu không đổi theo thời gian:
     $$u_t = rac{dx_t}{dt} = x_1 - x_0$$
   - Mạng neural $v_	heta(x_t, t, 	ext{condition})$ chỉ cần học dự đoán vector vận tốc này bằng hàm mất mát MSE cực kỳ ổn định:
     $$\mathcal{L}_{	ext{CFM}} = \mathbb{E}_{t, x_0, x_1} \left\| v_	heta(x_t, t, 	ext{condition}) - (x_1 - x_0) ight\|^2$$
   - Khi suy luận (Inference), bộ giải phương trình vi phân (Euler ODE Solver) chỉ cần **16 đến 32 bước** đi thẳng là đạt chất lượng âm thanh hoàn hảo!


In [ ]:
# Code mô phỏng: Conditional Flow Matching (CFM) Decoder hoàn chỉnh có Euler ODE Solver
class MinimalCFMDecoder(nn.Module):
    """
    Lấy cảm hứng từ Voicebox (Meta 2023) và F5-TTS (2024).
    Mạng dự đoán trường vector vận tốc v_t(x_t | t, condition) để sinh Mel-spectrogram.
    """
    def __init__(self, n_mels=80, cond_dim=768, hidden_dim=256):
        super().__init__()
        # Kênh đầu vào: n_mels (x_t) + 1 (thời gian t) + cond_dim (vector ngữ nghĩa & speaker)
        self.input_proj = nn.Conv1d(n_mels + 1 + cond_dim, hidden_dim, kernel_size=3, padding=1)
        self.backbone = nn.Sequential(
            nn.GELU(),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.GELU()
        )
        self.output_proj = nn.Conv1d(hidden_dim, n_mels, kernel_size=3, padding=1)
        
    def forward(self, x_t, t, condition):
        """
        x_t:       [Batch, n_mels, T] (Mel đang bị nhiễu ở bước t)
        t:         [Batch] (Thời gian liên tục t in [0, 1])
        condition: [Batch, cond_dim, T] (Ngữ nghĩa đích + Giọng nói người nói)
        """
        B, _, T = x_t.shape
        t_embed = t.view(B, 1, 1).expand(B, 1, T)
        net_in = torch.cat([x_t, t_embed, condition], dim=1)
        h = self.input_proj(net_in)
        h = self.backbone(h)
        v_pred = self.output_proj(h) # Dự đoán vector vận tốc dx/dt
        return v_pred

    @torch.no_grad()
    def sample_euler_ode(self, condition, num_steps=16):
        """
        Euler ODE Solver: Tích phân số từ t=0 (Nhiễu trắng) đến t=1 (Mel hoàn chỉnh)
        x_{t+dt} = x_t + v_pred * dt
        """
        B, _, T = condition.shape
        # Bước 0: Bắt đầu từ nhiễu thuần túy Gaussian N(0, I)
        x_t = torch.randn(B, 80, T, device=condition.device)
        dt = 1.0 / num_steps
        
        for step in range(num_steps):
            current_t = torch.full((B,), step * dt, device=condition.device)
            v_pred = self.forward(x_t, current_t, condition)
            x_t = x_t + v_pred * dt # Bước nhảy Euler
            
        return x_t # x_1 chính là Mel-spectrogram hoàn chỉnh!

# Chạy thử nghiệm CFM Decoder
cfm_decoder = MinimalCFMDecoder(n_mels=80, cond_dim=768, hidden_dim=128).to(device)
fake_condition = torch.randn(2, 768, 64, device=device) # 64 frames condition

# Sinh Mel-spectrogram trong 16 bước
generated_mel = cfm_decoder.sample_euler_ode(fake_condition, num_steps=16)

print("Kích thước Điều kiện (Whisper + Speaker Prompt):", fake_condition.shape)
print("Kích thước Mel sinh ra từ Euler ODE (16 bước):   ", generated_mel.shape)
print("-> Non-Autoregressive hoàn toàn: Không lo nuốt từ, không lo lặp chữ vô tận!")
